
# Decision Tree Regression on Tips Dataset  
### Cross-Validation & Hyperparameter Optimization (GridSearchCV)

**Objective**  
Build and evaluate a Decision Tree Regressor on the Tips dataset using:
- Cross-validation
- GridSearchCV for hyperparameter tuning
- Regression metrics
- Tree visualization

This notebook uses **pandas 3+**, **scikit-learn**, **seaborn**, and **matplotlib**.



## 1. Load Libraries


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



## 2. Load Tips Dataset

The Tips dataset is a classic regression dataset where:
- **Target**: `tip`
- **Features**: total_bill, sex, smoker, day, time, size


In [ ]:

df = sns.load_dataset("tips")
df.head()



## 3. Data Preprocessing

- Convert categorical variables using one-hot encoding
- Separate features (X) and target (y)


In [ ]:

df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop("tip", axis=1)
y = df_encoded["tip"]

X.head()



## 4. Train-Test Split


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



## 5. Baseline Decision Tree Regressor


In [ ]:

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred = dt.predict(X_test)



## 6. Evaluation Metrics

- **MAE**: Average absolute error  
- **MSE**: Penalizes larger errors  
- **R²**: Variance explained by the model


In [ ]:

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 :", r2_score(y_test, y_pred))



## 7. Cross-Validation Explained

**Cross-validation**:
- Splits training data into *k* folds (here k=5)
- Trains the model k times
- Each time uses a different fold as validation
- Final score is the average

This reduces overfitting and gives a more reliable performance estimate.


In [ ]:

cv_scores = cross_val_score(
    dt, X_train, y_train,
    cv=5,
    scoring="neg_mean_squared_error"
)

print("CV MSE Scores:", -cv_scores)
print("Average CV MSE:", -cv_scores.mean())



## 8. Hyperparameter Optimization using GridSearchCV

We tune:
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`

GridSearchCV:
- Tries all parameter combinations
- Uses cross-validation internally
- Selects the best model based on CV score


In [ ]:

param_grid = {
    "max_depth": [2, 3, 4, 5, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

grid.best_params_



## 9. Best Model Evaluation


In [ ]:

best_dt = grid.best_estimator_
y_pred_best = best_dt.predict(X_test)

print("Optimized MAE:", mean_absolute_error(y_test, y_pred_best))
print("Optimized MSE:", mean_squared_error(y_test, y_pred_best))
print("Optimized R2 :", r2_score(y_test, y_pred_best))



## 10. Decision Tree Visualization


In [ ]:

plt.figure(figsize=(18, 8))
plot_tree(
    best_dt,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.show()



## 11. Key Takeaways

- Decision Trees can easily overfit without constraints
- Cross-validation gives a stable performance estimate
- GridSearchCV finds optimal complexity
- Pruning parameters (`max_depth`, `min_samples_leaf`) are critical
